# Men's ice hockey olympics 2026 prediction

# 1. Importing libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, mean_absolute_error, confusion_matrix, recall_score, precision_score, f1_score

from sklearn.pipeline import Pipeline

# 2. Loading data

In this notebook, we use two datasets:

`ath_base.csv`: athlete-level historical data

`res_base.csv`: official results table (used for validation years 2018 and 2022)

The athlete dataset allows detailed filtering by sport and event.

The results dataset is used to validate and patch specific Olympic years for testing purposes.

In [2]:
ath_path = "ath_base.csv"
ath = pd.read_csv(ath_path)

In [3]:
ath.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


# 3. Filtering hockey men only

In this step, we filter the athlete dataset to keep only:

*  Winter olympics
*  Men's ice hockey

This ensures the model focuses only on the specific sport we want to predict, and other sports and events are removed.

In [4]:
hky_m = ath[
    (ath["Season"] == "Winter") &
    (ath["Event"] == "Ice Hockey Men's Ice Hockey")
].copy()

In [5]:
hky_m.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
28,9,Antti Sami Aalto,M,26.0,186.0,96.0,Finland,FIN,2002 Winter,2002,Winter,Salt Lake City,Ice Hockey,Ice Hockey Men's Ice Hockey,NaN
40,16,Juhamatti Tapio Aaltonen,M,28.0,184.0,85.0,Finland,FIN,2014 Winter,2014,Winter,Sochi,Ice Hockey,Ice Hockey Men's Ice Hockey,Bronze
672,391,Clarence John Abel,M,23.0,185.0,102.0,United States,USA,1924 Winter,1924,Winter,Chamonix,Ice Hockey,Ice Hockey Men's Ice Hockey,Silver
673,392,George Gordon Abel,M,35.0,NaN,NaN,Canada,CAN,1952 Winter,1952,Winter,Oslo,Ice Hockey,Ice Hockey Men's Ice Hockey,Gold
923,523,Trond Sevg Abrahamsen,M,19.0,183.0,87.0,Norway,NOR,1980 Winter,1980,Winter,Lake Placid,Ice Hockey,Ice Hockey Men's Ice Hockey,NaN


# 4. lineage merge (treat historical NOCs as the same performance team)

## 4.1 lineage merge

Some countries changed codes over time (splits / reunification).
If we keep them separate, lag features break and the model loses the real history.

So here, we map historical NOCs into one performance entity before building the base table.

In [6]:
#keep original noc for traceability
hky_m["NOC_original"] = hky_m["NOC"]

#define lineage groups
germany_lineage = ["FRG", "GDR", "GER"]
cze_lineage = ["TCH", "CZE", "SVK"]
russia_lineage = ["URS", "EUN", "RUS", "OAR", "ROC"]

#apply replacements
hky_m["NOC"] = hky_m["NOC"].replace(germany_lineage, "GER_ALL")
hky_m["NOC"] = hky_m["NOC"].replace(cze_lineage, "CZE_SVK_ALL")
hky_m["NOC"] = hky_m["NOC"].replace(russia_lineage, "RUS_ALL")

#quick verification
print("unique NOCs after lineage merge:")
display(sorted(hky_m["NOC"].unique()))

print("\nchecking mapping examples:")
display(
    hky_m[
        hky_m["NOC_original"].isin(
            germany_lineage + cze_lineage + russia_lineage
        )
    ][["Year", "NOC_original", "NOC"]]
    .drop_duplicates()
    .sort_values(["NOC_original", "Year"])
    .head(20)
)


unique NOCs after lineage merge:


['AUS',
 'AUT',
 'BEL',
 'BLR',
 'BUL',
 'CAN',
 'CZE_SVK_ALL',
 'FIN',
 'FRA',
 'GBR',
 'GER_ALL',
 'HUN',
 'ITA',
 'JPN',
 'KAZ',
 'LAT',
 'NED',
 'NOR',
 'POL',
 'ROU',
 'RUS_ALL',
 'SLO',
 'SUI',
 'SWE',
 'UKR',
 'USA',
 'YUG']


checking mapping examples:


,Year,NOC_original,NOC
4803,1994,CZE,CZE_SVK_ALL
19432,1998,CZE,CZE_SVK_ALL
33617,2002,CZE,CZE_SVK_ALL
31843,2006,CZE,CZE_SVK_ALL
23181,2010,CZE,CZE_SVK_ALL
14600,2014,CZE,CZE_SVK_ALL
16447,1992,EUN,RUS_ALL
12271,1968,FRG,GER_ALL
8322,1972,FRG,GER_ALL
10939,1976,FRG,GER_ALL


# 5. Creating unique medal table

we need to make sure that each medal is counted only once per country per Olympic year.

In the dataset, medals appear once for every player on the team and, since hockey teams have many players, one gold medal could appear several times.

To solve that:

*    remove rows where Medal is missing
*    remove duplicate rows using: Year, NOC (country), Medal, Event

With that, if Canada wins Gold in 2014, it is counted once, instead of once per player.

In [7]:
#each medal should be counted once per NOC per year (not once per athlete)
hky_m_unique_medals = (
    hky_m.dropna(subset=["Medal"])
         .drop_duplicates(subset=["Year", "NOC", "Medal", "Event"])
         .copy()
)

In [8]:
hky_m_unique_medals.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal,NOC_original
40,16,Juhamatti Tapio Aaltonen,M,28.0,184.0,85.0,Finland,FIN,2014 Winter,2014,Winter,Sochi,Ice Hockey,Ice Hockey Men's Ice Hockey,Bronze,FIN
672,391,Clarence John Abel,M,23.0,185.0,102.0,United States,USA,1924 Winter,1924,Winter,Chamonix,Ice Hockey,Ice Hockey Men's Ice Hockey,Silver,USA
673,392,George Gordon Abel,M,35.0,NaN,NaN,Canada,CAN,1952 Winter,1952,Winter,Oslo,Ice Hockey,Ice Hockey Men's Ice Hockey,Gold,CAN
935,529,Karl Gustaf Emanuel Abrahamsson,M,31.0,NaN,NaN,Sweden,SWE,1928 Winter,1928,Winter,Sankt Moritz,Ice Hockey,Ice Hockey Men's Ice Hockey,Silver,SWE
1962,1085,Maksim Sergeyevich Afinogenov,M,22.0,183.0,88.0,Russia,RUS_ALL,2002 Winter,2002,Winter,Salt Lake City,Ice Hockey,Ice Hockey Men's Ice Hockey,Bronze,RUS


# 6. Creating participation table (who played men's hockey in each olympics)

Considering that not every country who plays wins a medal, if we only keep medal data, we would be ignoring teams that played but did not win any medal.

Since this would create bias, we decided to:

*   remove duplicate combinations of Year and NOC
*   keep only those two columns
*   create a new column called participated and set it to 1

With that, we have a table where:

*   Each row represents one country in one olympic year that participated in men's hockey

This allows us to assign:

*   0 points for teams that played but did not win medals

Without this step, the model would have thought that only the medal winning teams exist.

In [9]:
hky_m_participation = (
    hky_m.drop_duplicates(subset=["Year", "NOC"])[["Year", "NOC"]]
         .assign(participated=1)
         .copy()
)

# 7. Mapping medals to points (creating hockey strength signal)

Converting medal types into numeric points.

*   Gold = 3
*   Silver = 2
*   Bronze = 1

This way, the model can learn that

*   Higher points = stronger performance
*   Zero points = no medal

This allows us to measure team performance over time in a consistent and numeric way. We will then use these points to create lag features and train prediction models.

In [10]:
medal_points_map = {"Gold": 3, "Silver": 2, "Bronze": 1}
hky_m_unique_medals["medal_points"] = hky_m_unique_medals["Medal"].map(medal_points_map).astype(int)

# 8. Building per NOC-year target table

Building the main working table for the model.

First, we group the medal data by:
*  Year
*  NOC (country)

Then we sum the medal points for each country in each Olympic year, which gives us:
*  3 points if the team won Gold
*  2 points for Silver
*  1 point for Bronze

Next, we merge this medal table with the participation table because it includes all teams that played, even if they did not win any medal.

After merging:
*   Teams that did not medal have NaN
* we replace NaN values with 0

WIth that, every team that played has a number of points (0, 1, 2, or 3)

Then, we have: `medal_flag`


This is:
*  1 if points > 0 (they won a medal)
*  0 if points = 0 (they did not medal)


In [11]:
#points=0 for most nocs, 3/2/1 for medalists
points_by_noc_year = (
    hky_m_unique_medals.groupby(["Year", "NOC"], as_index=False)["medal_points"]
                      .sum()
                      .rename(columns={"medal_points": "points"})
)

base = (
    hky_m_participation.merge(points_by_noc_year, on=["Year", "NOC"], how="left")
                       .fillna({"points": 0})
                       .copy()
)

base["points"] = base["points"].astype(int)
base["medal_flag"] = (base["points"] > 0).astype(int)

# 9. patching 2018 and 2022 from res_base

For testing purposes, our instructor Nasimeh Asgarian recommended to use 2018 and 2022 as validation years.

The athlete dataset already contains medal information, but we use res_base.csv as the official results reference for these specific years because:

*  The testing years reflect official final results
*  we want no inconsistencies between training and evaluation

Next, we filter res_base for: winter olympics, men's ice hockey, years 2018 and 2022, this results in:

*  creating a unique medal table (one medal per country per year)
*  mapping medals to points (gold=3,silver=2, bronze=1)
*  merging values into the main base table

For 2018 and 2022 only, we replace the existing points with the official results from res_base, rebuild the medal_flag column, to make sure that the model is trained on past years, and 2018/2022 are clean validation targets.

## 9.1 loading res_base

In [12]:
res_path = "res_base.csv"
res = pd.read_csv(res_path)

## 9.2 filtering men's ice hockey only (winter)

In [13]:
#note: res dataset uses different column names from ath

res_hky_m = res[
    (res["Season"] == "Winter") &
    (res["discipline"].str.contains("Ice Hockey", case=False, na=False)) &
    (res["event"].str.contains("Ice Hockey, Men", case=False, na=False))
].copy()

## 9.3 keeping only the years we want to patch/test

In [14]:
test_years = [2018, 2022]

res_hky_m = res_hky_m[res_hky_m["Year"].isin(test_years)].copy()

## 9.4 creating unique medal table in res

In [15]:
#(team medal should be counted once per NOC per year)
#res is also athlete lvl, so we must deduplicate

res_hky_m_medals = (
    res_hky_m.dropna(subset=["medal"])
             .drop_duplicates(subset=["Year", "NOC", "medal", "event"])
             .copy()
)

## 9.5 mapping medals to points

In [16]:
medal_points_map = {"Gold": 3, "Silver": 2, "Bronze": 1}
res_hky_m_medals["medal_points"] = res_hky_m_medals["medal"].map(medal_points_map).astype(int)

## 9.6 creating points per NOC-year from res

In [17]:
res_points_by_noc_year = (
    res_hky_m_medals.groupby(["Year", "NOC"], as_index=False)["medal_points"]
                    .sum()
                    .rename(columns={"medal_points": "points_res"})
)

## 9.7 building base_test from res (2018 and 2022)

In [18]:
#ath base ends at 2014, so we cannot patch rows that do not exist

#participation table in res (teams that played men's hockey that year)
res_participation = (
    res_hky_m.drop_duplicates(subset=["Year", "NOC"])[["Year", "NOC"]]
            .assign(participated=1)
            .copy()
)

#building test base table (include 0 points for non-medal teams)
base_test = (
    res_participation.merge(res_points_by_noc_year, on=["Year", "NOC"], how="left")
                     .fillna({"points_res": 0})
                     .rename(columns={"points_res": "points"})
                     .copy()
)

base_test["points"] = base_test["points"].astype(int)
base_test["medal_flag"] = (base_test["points"] > 0).astype(int)

print("base_test years:", sorted(base_test["Year"].unique()))
print("base_test shape:", base_test.shape)

base_test years: [np.int64(2018), np.int64(2022)]
base_test shape: (24, 5)


## 9.8 checking Results

In [19]:
#checking training base (ath) years
print("train years in base (last 12):", sorted(base["Year"].unique())[-12:])

#checking test base (res) years
print("test years in base_test:", sorted(base_test["Year"].unique()))

train years in base (last 12): [np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014)]
test years in base_test: [np.int64(2018), np.int64(2022)]


# 10. building full timeline

In [20]:
#combine ath base with res test years

full_base = pd.concat([base, base_test], ignore_index=True)

#sort chronologically by country and year
full_base = full_base.sort_values(["NOC", "Year"]).reset_index(drop=True)

print("years in full_base:", sorted(full_base["Year"].unique()))

years in full_base: [np.int64(1924), np.int64(1928), np.int64(1932), np.int64(1936), np.int64(1948), np.int64(1952), np.int64(1956), np.int64(1960), np.int64(1964), np.int64(1968), np.int64(1972), np.int64(1976), np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]


# 10. sorting for time-based feature creation

Before creating lag features, sorting the table by noc and year, because lag features look at previous Olympics.

If the data is not sorted in time order, the model might:

*  Look at the wrong previous year
*  Mix up history
*  Create incorrect lag values

Sorting makes sure that:

*  For each country, the Olympics are in chronological order.
*  This makes the time based features accurate.
*  Without sorting, the lag calculations would be wrong.

In [21]:
#sorting full timeline before creating lag features
full_base = full_base.sort_values(["NOC", "Year"]).reset_index(drop=True)

# 11. creating lag features (using past olympics performance)

In this step, we create lag features to help the model look at past performance to predict the future.

Instead of only asking: *"How many points does a team have?”*, we ask

*  how many points did the team have in the last olympics?
*  how many points two olympics ago?
*  what has been their recent average?

This helps the model understand momentum

What we create:

*  points_lag1: points in the previous Olympics
*  points_lag2: points two Olympics ago

*  medal_lag1: did they win medal in the previous Olympics?
*  medal_lag2: did they win medal two Olympics ago?

Then, we create rolling features:
*  points_roll3_mean: average points over the last 3 Olympics
*  medal_roll3_sum: how many times they medaled in the last 3 Olympics

We do that because hockey performance is not random, and strong teams tend to continue strong across multiple Olympic cycles. With these features we allow the model to learn performance trends instead of treating each year independently.

In [22]:
full_base["points_lag1"] = full_base.groupby("NOC")["points"].shift(1)
full_base["points_lag2"] = full_base.groupby("NOC")["points"].shift(2)

full_base["medal_lag1"] = full_base.groupby("NOC")["medal_flag"].shift(1)
full_base["medal_lag2"] = full_base.groupby("NOC")["medal_flag"].shift(2)

full_base["points_roll3_mean"] = (
    full_base.groupby("NOC")["points"]
             .shift(1)
             .rolling(window=3, min_periods=1)
             .mean()
             .reset_index(level=0, drop=True)
)

full_base["medal_roll3_sum"] = (
    full_base.groupby("NOC")["medal_flag"]
             .shift(1)
             .rolling(window=3, min_periods=1)
             .sum()
             .reset_index(level=0, drop=True)
)

# 12. filling missing lags (new or rare teams)

Some teams are new and some teams did not participate in previous olympics, so, when that happens, lag values become missing (NaN).

So, if a team appears for the first time, it does not have a previous Olympics.And instead of leaving these values missing, we replace them with 0

The reasoning behind it is that no past participation means no past points, and 0 is a logical and consistent value to keep the dataset clean and prevents model errors.

In [23]:
lag_cols = ["points_lag1", "points_lag2", "medal_lag1", "medal_lag2", "points_roll3_mean", "medal_roll3_sum"]

full_base[lag_cols] = full_base[lag_cols].fillna(0)

# 13. preparing training data (up to last known olympics in dataset)

In this step, we prepare the data that will be used to train the model.

1. identify the most recent olympic year available in the dataset.
2. create a training dataset using only years before 2026

Why?

Because we are predicting 2026.

If I included 2026 in training, that would be data leakage.
The model must only learn from past data, and because the tournament is just starting as of now.

In [24]:
#preparing training and test data (train <=2014, test = 2018 and 2022)
train_end_year = 2014
test_years = [2018, 2022]

#training split (all years up to 2014)
train = full_base[full_base["Year"] <= train_end_year].copy()

#test split (2018 and 2022)
test = full_base[full_base["Year"].isin(test_years)].copy()

#features (host feature excluded for this evaluation)
feature_cols = [
    "points_lag1", "points_lag2",
    "medal_lag1", "medal_lag2",
    "points_roll3_mean", "medal_roll3_sum"
]

#training arrays
X_train = train[feature_cols].copy()
y_train_medal = train["medal_flag"].copy()
y_train_points = train["points"].copy()

print("train years:", sorted(train["Year"].unique())[-10:])
print("test years:", sorted(test["Year"].unique()))
print("train shape:", train.shape, "| test shape:", test.shape)

train years: [np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014)]
test years: [np.int64(2018), np.int64(2022)]
train shape: (245, 11) | test shape: (24, 11)


In [25]:
#checking
print("non-zero lag share in test:")
display((test[feature_cols] != 0).mean().to_frame("share_nonzero").T)

non-zero lag share in test:


,points_lag1,points_lag2,medal_lag1,medal_lag2,points_roll3_mean,medal_roll3_sum
share_nonzero,0.25,0.25,0.25,0.25,0.583333,0.583333


# 14. training models

the 2 predictions will be combined into a ranking score

## 14.1: probability of winning any medal (yes/no)

Predicts the probability that a team wins any medal.

In [26]:
#using class_weight balanced due to heavy imbalance (mostly zeros)
medal_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=500, random_state=42))
])

medal_model.fit(X_train, y_train_medal)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=500,
                                    random_state=42))])

## 14.2: expected points (0/1/2/3)

Predicts expected medal strength (0–3 scale)

In [27]:
#using poisson regressor (count like target)

points_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pois", PoissonRegressor(alpha=0.01, max_iter=1000))
])

points_model.fit(X_train, y_train_points)

Pipeline(steps=[('scaler', StandardScaler()),
                ('pois', PoissonRegressor(alpha=0.01, max_iter=1000))])

## 14.3: checking

In [28]:
print("train rows:", X_train.shape[0])
print("train years:", sorted(train["Year"].unique())[-10:])
print("positive medal rate (train):", y_train_medal.mean())
print("avg points (train):", y_train_points.mean())

print("\ntest years:", sorted(test["Year"].unique()))
print("test rows:", test.shape[0])

#confirming test has signal (not all zeros anymore)
print("\nnon-zero lag share in test:")
display((test[feature_cols] != 0).mean().to_frame("share_nonzero").T)

train rows: 245
train years: [np.int64(1980), np.int64(1984), np.int64(1988), np.int64(1992), np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014)]
positive medal rate (train): 0.2693877551020408
avg points (train): 0.5387755102040817

test years: [np.int64(2018), np.int64(2022)]
test rows: 24

non-zero lag share in test:


,points_lag1,points_lag2,medal_lag1,medal_lag2,points_roll3_mean,medal_roll3_sum
share_nonzero,0.25,0.25,0.25,0.25,0.583333,0.583333


# 15. Models evaluation

## 15.1 test years and eval function

In [29]:
def evaluate_year(df_test, year):
    df_y = df_test[df_test["Year"] == year].copy()

    X_test = df_y[feature_cols].copy()
    y_true_medal = df_y["medal_flag"].copy()
    y_true_points = df_y["points"].copy()

    df_y["pred_p_medal"] = medal_model.predict_proba(X_test)[:, 1]
    df_y["pred_points"] = points_model.predict(X_test)
    df_y["pred_points"] = np.clip(df_y["pred_points"], 0, 3)

    df_y["pred_medal_flag"] = (df_y["pred_p_medal"] >= 0.5).astype(int)

    acc = accuracy_score(y_true_medal, df_y["pred_medal_flag"])
    mae = mean_absolute_error(y_true_points, df_y["pred_points"])
    cm = confusion_matrix(y_true_medal, df_y["pred_medal_flag"])

    print("evaluation year:", year)
    print("teams in test year:", df_y.shape[0])
    print("medal accuracy:", round(acc, 3))
    print("points MAE:", round(mae, 3))
    print("confusion matrix:\n", cm)

    df_y["score"] = df_y["pred_p_medal"] * df_y["pred_points"]

    pred_top3 = (
        df_y.sort_values(["score", "pred_points", "pred_p_medal"], ascending=False)
            .head(3)
            [["NOC", "pred_p_medal", "pred_points", "score"]]
            .copy()
    )

    actual_medalists = (
        df_y[df_y["points"] > 0]
            .sort_values("points", ascending=False)
            [["NOC", "points"]]
            .copy()
    )

    print("\npredicted top 3:")
    display(pred_top3)

    print("actual medalists:")
    display(actual_medalists)

    hits = len(set(pred_top3["NOC"]).intersection(set(actual_medalists["NOC"])))
    print("top3 overlap hits:", hits, "/ 3")

    return df_y

## 15.2 running evaluation for both years

In [30]:
eval_2018 = evaluate_year(test, 2018)

evaluation year: 2018
teams in test year: 12
medal accuracy: 0.583
points MAE: 0.73
confusion matrix:
 [[5 4]
 [1 2]]

predicted top 3:


,NOC,pred_p_medal,pred_points,score
41,CAN,0.895668,1.441972,1.291528
84,FIN,0.853324,1.194181,1.019023
262,USA,0.708453,0.845982,0.599339


actual medalists:


,NOC,points
175,ROC,3
99,GER,2
41,CAN,1


top3 overlap hits: 1 / 3


In [31]:
eval_2022 = evaluate_year(test, 2022)

evaluation year: 2022
teams in test year: 12
medal accuracy: 0.583
points MAE: 0.864
confusion matrix:
 [[5 4]
 [1 2]]

predicted top 3:


,NOC,pred_p_medal,pred_points,score
42,CAN,0.957422,2.023274,1.937126
100,GER,0.795202,0.896859,0.713184
85,FIN,0.725424,0.931383,0.675648


actual medalists:


,NOC,points
85,FIN,3
176,ROC,2
217,SVK,1


top3 overlap hits: 1 / 3


# 16. Tuning threshold

## 16.1 Collect predictions for both test years

In [32]:
test_eval = test.copy()

test_eval["pred_p_medal"] = medal_model.predict_proba(test_eval[feature_cols])[:, 1]
test_eval["pred_points"] = points_model.predict(test_eval[feature_cols])
test_eval["pred_points"] = np.clip(test_eval["pred_points"], 0, 3)

print("total test rows:", test_eval.shape[0])
print("true medal rate:", test_eval["medal_flag"].mean())

total test rows: 24
true medal rate: 0.25


## 16.2 Evaluate multiple thresholds

In [33]:
thresholds = np.linspace(0.1, 0.9, 17)

results = []

for t in thresholds:
    preds = (test_eval["pred_p_medal"] >= t).astype(int)

    acc = accuracy_score(test_eval["medal_flag"], preds)
    recall = recall_score(test_eval["medal_flag"], preds)
    precision = precision_score(test_eval["medal_flag"], preds, zero_division=0)
    f1 = f1_score(test_eval["medal_flag"], preds)

    results.append({
        "threshold": t,
        "accuracy": acc,
        "recall": recall,
        "precision": precision,
        "f1": f1
    })

threshold_df = pd.DataFrame(results).sort_values("f1", ascending=False)

display(threshold_df.head(10))

,threshold,accuracy,recall,precision,f1
7,0.45,0.583333,0.666667,0.333333,0.444444
8,0.50,0.583333,0.666667,0.333333,0.444444
0,0.10,0.250000,1.000000,0.250000,0.400000
1,0.15,0.250000,1.000000,0.250000,0.400000
2,0.20,0.250000,1.000000,0.250000,0.400000
4,0.30,0.500000,0.666667,0.285714,0.400000
3,0.25,0.500000,0.666667,0.285714,0.400000
6,0.40,0.500000,0.666667,0.285714,0.400000
5,0.35,0.500000,0.666667,0.285714,0.400000
9,0.55,0.583333,0.500000,0.300000,0.375000


We will keep the threshold at 0.5 because it gives a balanced decision between predicting too many teams as medalists and being too conservative, while keeping our results stable and easy to interpret.

# 17. predictions 2026

We did not include a host advantage feature, as the model was trained purely on historical performance trends. Host adjustment can be considered as a future enhancement.

## 17.1 building final training set for 2026 prediction (train <= 2022)

Now that we validated the approach using 2018 and 2022, we retrain the models using all data available up to 2022.  
This gives the model the most recent signal before predicting 2026.

In [34]:
#training end year for final 2026 prediction
final_train_end_year = 2022

#final training table (use everything up to 2022)
train_2026 = full_base[full_base["Year"] <= final_train_end_year].copy()

#features (same as evaluation, no host feature here)
feature_cols_2026 = [
    "points_lag1", "points_lag2",
    "medal_lag1", "medal_lag2",
    "points_roll3_mean", "medal_roll3_sum"
]

X_train_2026 = train_2026[feature_cols_2026].copy()
y_train_medal_2026 = train_2026["medal_flag"].copy()
y_train_points_2026 = train_2026["points"].copy()

print("train_2026 years:", sorted(train_2026["Year"].unique())[-8:])
print("train_2026 shape:", train_2026.shape)
print("medal rate:", y_train_medal_2026.mean())

train_2026 years: [np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]
train_2026 shape: (269, 11)
medal rate: 0.26765799256505574


## 17.2 training final models for 2026 (using data up to 2022)

We retrain both models:
- medal model: probability of winning any medal (yes/no)
- points model: expected medal points (0 to 3)

These models will be used to create the 2026 ranking.

In [35]:
#medal probability model
medal_model_2026 = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=500, random_state=42))
])

medal_model_2026.fit(X_train_2026, y_train_medal_2026)

#expected points model
points_model_2026 = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pois", PoissonRegressor(alpha=0.01, max_iter=1000))
])

points_model_2026.fit(X_train_2026, y_train_points_2026)

print("models trained for 2026 prediction")

models trained for 2026 prediction


## 17.3 creating the 2026 prediction frame (teams we will predict)

We create one row per team (NOC) for 2026.  
Then we attach the latest available lag/rolling features from 2022 for each team.

In [36]:
pred_year = 2026

#teams to predict = teams that exist in the most recent year we have (2022)
latest_year_for_roster = int(full_base["Year"].max())
teams_2026 = sorted(full_base[full_base["Year"] == latest_year_for_roster]["NOC"].unique())

pred_frame_2026 = pd.DataFrame({
    "Year": [pred_year] * len(teams_2026),
    "NOC": teams_2026
})

#taking last known feature row per team (from <=2022)
last_rows = (
    full_base[full_base["Year"] <= final_train_end_year]
        .sort_values(["NOC", "Year"])
        .groupby("NOC")
        .tail(1)
        [["NOC"] + feature_cols_2026]
        .copy()
)

pred_2026 = pred_frame_2026.merge(last_rows, on="NOC", how="left")

#safety: teams with missing history get 0s
pred_2026[feature_cols_2026] = pred_2026[feature_cols_2026].fillna(0)

print("2026 prediction frame shape:", pred_2026.shape)
display(pred_2026.head())

2026 prediction frame shape: (12, 8)


,Year,NOC,points_lag1,points_lag2,medal_lag1,medal_lag2,points_roll3_mean,medal_roll3_sum
0,2026,CAN,1.0,3.0,1.0,1.0,2.333333,3.0
1,2026,CHN,0.0,0.0,0.0,0.0,2.000000,2.0
2,2026,CZE,0.0,0.0,0.0,0.0,0.000000,0.0
3,2026,DEN,0.0,0.0,0.0,0.0,0.500000,1.0
4,2026,FIN,0.0,1.0,0.0,1.0,0.666667,2.0


## 17.4 predicting 2026 medal probability + expected points and ranking teams

We predict:
- p_medal: probability of winning any medal
- exp_points: expected medal points (0–3)
Then we combine them into one score to rank teams:
score = p_medal * exp_points

In [37]:
#predicting medal probability
pred_2026["p_medal"] = medal_model_2026.predict_proba(pred_2026[feature_cols_2026])[:, 1]

#predicting expected points
pred_2026["exp_points"] = points_model_2026.predict(pred_2026[feature_cols_2026])
pred_2026["exp_points"] = np.clip(pred_2026["exp_points"], 0, 3)

#combining into ranking score
pred_2026["score"] = pred_2026["p_medal"] * pred_2026["exp_points"]

#ranking
pred_ranked_2026 = (
    pred_2026.sort_values(["score", "exp_points", "p_medal"], ascending=False)
             .reset_index(drop=True)
             .copy()
)

print("men's ice hockey 2026 predicted ranking (top 12)")
display(pred_ranked_2026.head(12)[["NOC", "p_medal", "exp_points", "score"]])

men's ice hockey 2026 predicted ranking (top 12)


,NOC,p_medal,exp_points,score
0,CAN,0.917611,1.515618,1.390747
1,FIN,0.697413,0.945136,0.659150
2,GER,0.741214,0.815813,0.604692
3,SWE,0.699803,0.859814,0.601700
4,CHN,0.669399,0.760966,0.509390
5,ROC,0.586293,0.506460,0.296934
6,USA,0.406827,0.351990,0.143199
7,DEN,0.380374,0.320051,0.121739
8,CZE,0.265767,0.238199,0.063306
9,LAT,0.265767,0.238199,0.063306


## 17.5 predicted podium (top 3)

We take the top 3 teams from the ranking score and label them as:
Gold, Silver, Bronze.
This is a simple ranking-based podium (not a full tournament simulation).

In [38]:
top3_2026 = pred_ranked_2026.head(3).copy()
top3_2026["predicted_medal"] = ["Gold", "Silver", "Bronze"]

print("men's ice hockey 2026 predicted podium")
display(top3_2026[["NOC", "predicted_medal", "p_medal", "exp_points", "score"]])

men's ice hockey 2026 predicted podium


,NOC,predicted_medal,p_medal,exp_points,score
0,CAN,Gold,0.917611,1.515618,1.390747
1,FIN,Silver,0.697413,0.945136,0.659150
2,GER,Bronze,0.741214,0.815813,0.604692


# 18. tournament simulation (10,000 runs) to estimate medal probabilities

Ranking is useful, but hockey is a knockout-style tournament where upsets happen. So here, we simulate the tournament 10,000 times.

Each match outcome is random, but weighted by the predicted team strength.

At the end, we count how often each team wins:
- Gold
- Silver
- Bronze
- Any medal

## 18.1 selecting active teams and building bracket for simulation

In [39]:
#how many teams in the simulated bracket
#8 = clean quarterfinal bracket
top_teams_n = 8

sim_teams = (
    pred_ranked_2026.sort_values(["score", "exp_points", "p_medal"], ascending=False)
                    .head(top_teams_n)
                    .reset_index(drop=True)
                    .copy()
)

#creating strength signal
#score is p_medal * exp_points, but for match win prob we want a smoother rating
sim_teams["strength"] = (0.60 * sim_teams["p_medal"]) + (0.40 * sim_teams["exp_points"])

#temperature controls randomness (lower = favorites win more often)
temperature = 0.80

#standardizing strength into a rating space
sim_teams["rating"] = (sim_teams["strength"] - sim_teams["strength"].mean()) / (sim_teams["strength"].std() + 1e-9)
sim_teams["rating"] = sim_teams["rating"] / temperature

#lookup map: NOC -> rating
rating_map = dict(zip(sim_teams["NOC"], sim_teams["rating"]))

print("simulation teams (seed order):")
display(sim_teams[["NOC", "p_medal", "exp_points", "score", "strength", "rating"]])

simulation teams (seed order):


,NOC,p_medal,exp_points,score,strength,rating
0,CAN,0.917611,1.515618,1.390747,1.156813,2.284092
1,FIN,0.697413,0.945136,0.659150,0.796502,0.535299
2,GER,0.741214,0.815813,0.604692,0.771053,0.411781
3,SWE,0.699803,0.859814,0.601700,0.763807,0.376611
4,CHN,0.669399,0.760966,0.509390,0.706026,0.096166
5,ROC,0.586293,0.506460,0.296934,0.554360,-0.639956
6,USA,0.406827,0.351990,0.143199,0.384893,-1.462475
7,DEN,0.380374,0.320051,0.121739,0.356245,-1.601517


## 18.2 simulating matches using logistic win probability (bradley-terry)

In [40]:
def win_prob(team_a, team_b):
    ra = rating_map[team_a]
    rb = rating_map[team_b]
    return 1 / (1 + np.exp(-(ra - rb)))

def play_match(team_a, team_b, rng):
    p = win_prob(team_a, team_b)
    return team_a if rng.random() < p else team_b


#8-team seeded bracket:
#qf: 1v8, 2v7, 3v6, 4v5
#sf: winner(1v8) vs winner(4v5), winner(2v7) vs winner(3v6)
#bronze: semifinal losers


def simulate_one_tournament(seeds, rng):
    #quarterfinals
    qf1_w = play_match(seeds[0], seeds[7], rng)
    qf2_w = play_match(seeds[1], seeds[6], rng)
    qf3_w = play_match(seeds[2], seeds[5], rng)
    qf4_w = play_match(seeds[3], seeds[4], rng)

    #semifinals
    sf1_w = play_match(qf1_w, qf4_w, rng)
    sf2_w = play_match(qf2_w, qf3_w, rng)

    #semifinal losers (for bronze game)
    sf1_l = qf4_w if sf1_w == qf1_w else qf1_w
    sf2_l = qf3_w if sf2_w == qf2_w else qf2_w

    #final
    gold = play_match(sf1_w, sf2_w, rng)
    silver = sf2_w if gold == sf1_w else sf1_w

    #bronze match
    bronze = play_match(sf1_l, sf2_l, rng)

    return gold, silver, bronze

## 18.3 running tournament simulation many times

In [47]:
n_sims = 10_000

seeds = sim_teams["NOC"].tolist()

#medal counters
medal_counts = {t: {"Gold": 0, "Silver": 0, "Bronze": 0} for t in seeds}

#reproducible randomness
rng = np.random.default_rng(42)

for _ in range(n_sims):
    g, s, b = simulate_one_tournament(seeds, rng)
    medal_counts[g]["Gold"] += 1
    medal_counts[s]["Silver"] += 1
    medal_counts[b]["Bronze"] += 1

#converting counts to probabilities
rows = []
for t in seeds:
    rows.append({
        "NOC": t,
        "P_Gold": medal_counts[t]["Gold"] / n_sims,
        "P_Silver": medal_counts[t]["Silver"] / n_sims,
        "P_Bronze": medal_counts[t]["Bronze"] / n_sims,
        "P_Medal": (medal_counts[t]["Gold"] + medal_counts[t]["Silver"] + medal_counts[t]["Bronze"]) / n_sims
    })

sim_probs_2026 = (
    pd.DataFrame(rows)
      .sort_values("P_Medal", ascending=False)
      .reset_index(drop=True)
      .copy()
)

print("simulated medal probabilities (top teams)")
display(sim_probs_2026)

simulated medal probabilities (top teams)


,NOC,P_Gold,P_Silver,P_Bronze,P_Medal
0,CAN,0.7489,0.1176,0.1005,0.9670
1,FIN,0.1091,0.4073,0.1907,0.7071
2,GER,0.0719,0.3153,0.1837,0.5709
3,SWE,0.0387,0.0405,0.2775,0.3567
4,CHN,0.0237,0.0279,0.1807,0.2323
5,ROC,0.0067,0.0675,0.0498,0.1240
6,USA,0.0007,0.0215,0.0146,0.0368
7,DEN,0.0003,0.0024,0.0025,0.0052


## 18.4 canada medal probabilities from simulation

In [42]:
can_sim = sim_probs_2026[sim_probs_2026["NOC"] == "CAN"].copy()

print("canada simulated probabilities")
if len(can_sim) == 0:
    print("CAN is not in the selected bracket. increase top_teams_n and rerun simulation.")
else:
    display(can_sim)

canada simulated probabilities


,NOC,P_Gold,P_Silver,P_Bronze,P_Medal
0,CAN,0.7489,0.1176,0.1005,0.967


# Results

In [43]:
p = can_sim["P_Gold"].iloc[0]
n = n_sims

se = np.sqrt(p * (1 - p) / n)

lower = p - 1.96 * se
upper = p + 1.96 * se

print("Canada Gold Probability:", round(p,4))
print("95% CI:", (round(lower,4), round(upper,4)))

Canada Gold Probability: 0.7489
95% CI: (np.float64(0.7404), np.float64(0.7574))


Based on historical Olympic data (training through 2022) and 10,000 simulated Olympic brackets, our model estimates Canada has a 74.9% probability of winning gold and a 96.7% probability of earning any medal at the 2026 Winter Olympics. Finland and Germany follow as the next strongest projected contenders.

These estimates are generated using two statistical models: a Logistic Regression model to estimate medal probability and a Poisson regression model to estimate expected medal strength, combined into a tournament simulation framework. The gold probability estimate carries a narrow simulation-based 95% confidence interval of approximately 74.0%–75.7%, reflecting stability across simulation runs.

These projections are data-driven and based on historical performance trends, and do not account for future roster composition, NHL participation, injuries, or other real-world variables that may affect outcomes.

In [43]:
#Our last sentence, or perhaps the main sentence shold be regarding Canada odds,
#based on 100 years of data of medals, of course we have several constraints, but this is not a robust model
#do not use this information blindly to make bets
#this was a mini project for fun

#probability there's a 96% chance that canada will win a medal
#we have constraints for this, since this was a mini project.
#(we have an NDA in place)
#goal in this project